In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Required imports
import torch
from mapanything.models import MapAnything
from mapanything.utils.image import load_images

# Get inference device
device = "cuda" if torch.cuda.is_available() else "cpu"
device


'cuda'

In [2]:
# Init model - This requires internet access or the huggingface hub cache to be pre-downloaded
# For Apache 2.0 license model, use "facebook/map-anything-apache"
model = MapAnything.from_pretrained("facebook/map-anything").to(device)

Loading pretrained dinov2_vitg14 from torch hub


Using cache found in /home/soph/.cache/torch/hub/facebookresearch_dinov2_main


In [4]:
model

MapAnything(
  (encoder): DINOv2Encoder(
    (model): DinoVisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 1536, kernel_size=(14, 14), stride=(14, 14))
        (norm): Identity()
      )
      (blocks): ModuleList(
        (0-23): 24 x NestedTensorBlock(
          (norm1): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
          (attn): _AttentionWrapper(
            (qkv): Linear(in_features=1536, out_features=4608, bias=True)
            (proj): Linear(in_features=1536, out_features=1536, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): LayerScale()
          (drop_path1): Identity()
          (norm2): LayerNorm((1536,), eps=1e-06, elementwise_affine=True)
          (mlp): SwiGLUFFNFused(
            (w12): Linear(in_features=1536, out_features=8192, bias=True)
            (w3): Linear(in_features=4096, out_features=1536, bias=True)
          )
          (ls2): LayerScale()
          (drop_path2): Iden

In [14]:
# Load and preprocess images from a folder or list of paths
#images = ["images/pipeline-cat-chonk.jpeg"]  # or ["path/to/img1.jpg", "path/to/img2.jpg", ...]
#images = ["images/134.jpg"] 
images = "images/scene_112/impressionism"
views = load_images(images)

# Run inference
predictions = model.infer(
    views,                            # Input views
    memory_efficient_inference=True,  # Trades off speed for more views (up to 2000 views on 140 GB). Trade off is negligible - see profiling section
    minibatch_size=None,              # Minibatch size for memory-efficient inference (use 1 for smallest GPU memory consumption). Default is dynamic computation based on available GPU memory.
    use_amp=True,                     # Use mixed precision inference (recommended)
    amp_dtype="bf16",                 # bf16 inference (recommended; falls back to fp16 if bf16 not supported)
    apply_mask=True,                  # Apply masking to dense geometry outputs
    mask_edges=True,                  # Remove edge artifacts by using normals and depth
    apply_confidence_mask=False,      # Filter low-confidence regions
    confidence_percentile=10,         # Remove bottom 10 percentile confidence pixels
    use_multiview_confidence=False,   # Enable multi-view depth consistency based confidence in place of learning-based one
)
predictions

[{'pts3d': tensor([[[[-0.7435, -0.5715,  1.2385],
            [-0.7253, -0.5548,  1.2249],
            [-0.7429, -0.5740,  1.2532],
            ...,
            [ 0.7961, -0.6054,  1.3146],
            [ 0.0000, -0.0000,  0.0000],
            [ 0.0000, -0.0000,  0.0000]],
  
           [[-0.7293, -0.5596,  1.2272],
            [-0.7350, -0.5661,  1.2408],
            [-0.7372, -0.5686,  1.2430],
            ...,
            [ 0.7950, -0.6031,  1.3214],
            [ 0.0000, -0.0000,  0.0000],
            [ 0.0000, -0.0000,  0.0000]],
  
           [[-0.7282, -0.5519,  1.2193],
            [-0.7378, -0.5644,  1.2419],
            [-0.7344, -0.5641,  1.2363],
            ...,
            [ 0.7961, -0.6020,  1.3146],
            [ 0.7967, -0.5992,  1.3157],
            [ 0.8074, -0.6105,  1.3248]],
  
           ...,
  
           [[-0.8260,  0.6114,  1.3613],
            [-0.8249,  0.6052,  1.3591],
            [-0.8141,  0.5984,  1.3466],
            ...,
            [ 0.7958,  0.5907, 

In [15]:
import numpy as np
import torch
import open3d as o3d

all_points = []
all_colors = []

for pred in predictions:  # one dict per view
    pts3d = pred["pts3d"]
    img = pred["img_no_norm"]
    mask = pred.get("mask", None)

    if isinstance(pts3d, torch.Tensor):
        pts3d = pts3d.detach().float().cpu().numpy()
    if isinstance(img, torch.Tensor):
        img = img.detach().float().cpu().numpy()
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy()

    # remove batch dim if present (B,H,W,C)->(H,W,C), usually B=1
    if pts3d.ndim == 4:
        pts3d = pts3d[0]
    if img.ndim == 4:
        img = img[0]
    if mask is not None and mask.ndim == 4:
        mask = mask[0]

    points = pts3d.reshape(-1, 3)
    colors = img.reshape(-1, 3)

    valid = np.isfinite(points).all(axis=1)
    if mask is not None:
        valid = valid & (mask.reshape(-1) > 0)

    points = points[valid]
    colors = colors[valid]

    if colors.max() > 1.0:
        colors = colors / 255.0
    colors = np.clip(colors, 0.0, 1.0)

    all_points.append(points)
    all_colors.append(colors)

points_merged = np.concatenate(all_points, axis=0)
colors_merged = np.concatenate(all_colors, axis=0)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points_merged.astype(np.float64))
pcd.colors = o3d.utility.Vector3dVector(colors_merged.astype(np.float64))

print("Merged points:", np.asarray(pcd.points).shape[0])
o3d.visualization.draw_geometries([pcd], window_name="MapAnything Multi-view Point Cloud")

Merged points: 1157735


In [ ]:
# Access results for each view - Complete list of metric outputs
for i, pred in enumerate(predictions):
    # Geometry outputs
    pts3d = pred["pts3d"]                     # 3D points in world coordinates (B, H, W, 3)
    pts3d_cam = pred["pts3d_cam"]             # 3D points in camera coordinates (B, H, W, 3)
    depth_z = pred["depth_z"]                 # Z-depth in camera frame (B, H, W, 1)
    depth_along_ray = pred["depth_along_ray"] # Depth along ray in camera frame (B, H, W, 1)

    # Camera outputs
    ray_directions = pred["ray_directions"]   # Ray directions in camera frame (B, H, W, 3)
    intrinsics = pred["intrinsics"]           # Recovered pinhole camera intrinsics (B, 3, 3)
    camera_poses = pred["camera_poses"]       # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world poses in world frame (B, 4, 4)
    cam_trans = pred["cam_trans"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world translation in world frame (B, 3)
    cam_quats = pred["cam_quats"]             # OpenCV (+X - Right, +Y - Down, +Z - Forward) cam2world quaternion in world frame (B, 4)

    # Quality and masking
    confidence = pred["conf"]                 # Per-pixel confidence scores (B, H, W)
    mask = pred["mask"]                       # Combined validity mask (B, H, W, 1)
    non_ambiguous_mask = pred["non_ambiguous_mask"]                # Non-ambiguous regions (B, H, W)
    non_ambiguous_mask_logits = pred["non_ambiguous_mask_logits"]  # Mask logits (B, H, W)

    # Scaling
    metric_scaling_factor = pred["metric_scaling_factor"]  # Applied metric scaling (B,)

    # Original input
    img_no_norm = pred["img_no_norm"]         # Denormalized input images for visualization (B, H, W, 3)

## Visualize MapAnything output with Open3D

This cell converts `predictions[0]` into an Open3D point cloud and opens an interactive viewer.

If Open3D is not installed, run:
`%pip install open3d`

In [6]:
import numpy as np
import torch
import open3d as o3d

# Pick the first predicted view.
pred = predictions[0]

# Convert tensors to CPU numpy arrays.
pts3d = pred["pts3d"]
img = pred["img_no_norm"]
mask = pred.get("mask", None)

if isinstance(pts3d, torch.Tensor):
    pts3d = pts3d.detach().float().cpu().numpy()
if isinstance(img, torch.Tensor):
    img = img.detach().float().cpu().numpy()
if isinstance(mask, torch.Tensor):
    mask = mask.detach().cpu().numpy()

# Remove batch dimension if present: (1, H, W, C) -> (H, W, C).
if pts3d.ndim == 4:
    pts3d = pts3d[0]
if img.ndim == 4:
    img = img[0]
if mask is not None and mask.ndim == 4:
    mask = mask[0]

# Flatten to N x 3.
points = pts3d.reshape(-1, 3)
colors = img.reshape(-1, 3)

# Keep only valid points via mask (if available) and finite coordinates.
valid = np.isfinite(points).all(axis=1)
if mask is not None:
    valid = valid & (mask.reshape(-1) > 0)

# points = points[valid]
# colors = colors[valid]

# Ensure colors are in [0, 1].
if colors.max() > 1.0:
    colors = colors / 255.0
colors = np.clip(colors, 0.0, 1.0)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64))

# Optional cleanup for nicer visualization.
#pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)

print(f"Point cloud has {np.asarray(pcd.points).shape[0]} points")
o3d.visualization.draw_geometries([pcd], window_name="MapAnything Point Cloud")

Point cloud has 203056 points


In [28]:
import numpy as np
import torch
import open3d as o3d
import matplotlib.cm as cm

all_pcds = []
colormap = cm.get_cmap('hsv')

for view_idx, pred in enumerate(predictions):  # one dict per view
    pts3d = pred["pts3d"]
    img = pred["img_no_norm"]
    mask = pred.get("mask", None)

    if isinstance(pts3d, torch.Tensor):
        pts3d = pts3d.detach().float().cpu().numpy()
    if isinstance(img, torch.Tensor):
        img = img.detach().float().cpu().numpy()
    if isinstance(mask, torch.Tensor):
        mask = mask.detach().cpu().numpy()

    # remove batch dim if present (B,H,W,C)->(H,W,C), usually B=1
    if pts3d.ndim == 4:
        pts3d = pts3d[0]
    if img.ndim == 4:
        img = img[0]
    if mask is not None and mask.ndim == 4:
        mask = mask[0]

    points = pts3d.reshape(-1, 3)
    colors = img.reshape(-1, 3)

    valid = np.isfinite(points).all(axis=1)
    if mask is not None:
        valid = valid & (mask.reshape(-1) > 0)
    
    points = points[valid]
    colors = colors[valid]

    # Assign a unique color to this view from the colormap
    view_color = colormap(view_idx / len(predictions))[:3]  # RGB
    uniform_colors = np.full((len(points), 3), view_color)

    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
    pcd.colors = o3d.utility.Vector3dVector(uniform_colors.astype(np.float64))
    
    all_pcds.append(pcd)
    print(f"View {view_idx}: {len(points)} points, color: RGB{tuple(np.round(view_color, 2))}")

print(f"\nTotal PCDs: {len(all_pcds)}")
o3d.visualization.draw_geometries(all_pcds, window_name="MapAnything Multi-view Point Clouds - Color per View")


/tmp/ipykernel_4226/3251248032.py:7: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  colormap = cm.get_cmap('hsv')


View 0: 221834 points, color: RGB(np.float64(1.0), np.float64(0.0), np.float64(0.0))
View 1: 182563 points, color: RGB(np.float64(0.99), np.float64(0.96), np.float64(0.0))
View 2: 140788 points, color: RGB(np.float64(0.03), np.float64(1.0), np.float64(0.0))
View 3: 267273 points, color: RGB(np.float64(0.0), np.float64(1.0), np.float64(0.96))
View 4: 203563 points, color: RGB(np.float64(0.0), np.float64(0.06), np.float64(1.0))
View 5: 227728 points, color: RGB(np.float64(0.93), np.float64(0.0), np.float64(1.0))

Total PCDs: 6


In [9]:
from PIL import Image
import numpy as np
import open3d as o3d

# load pts (H,W,3) from file or prediction
pts3d = np.load("10002_pts3d.npy")
if pts3d.ndim == 4: pts3d = pts3d[0]
H, W = pts3d.shape[:2]

# load image and make sure it's RGB and same size as pts3d
img = Image.open("10002.jpg").convert("RGB")
if (img.height, img.width) != (H, W):
    img = img.resize((W, H), resample=Image.LANCZOS)

img_np = np.asarray(img)             # uint8 H x W x 3
img_np = img_np.astype(np.float32)
if img_np.max() > 1.0:
    img_np /= 255.0                  # normalize to [0,1]
img_np = np.clip(img_np, 0.0, 1.0)

# flatten and filter by valid points (use your mask if available)
points = pts3d.reshape(-1, 3)
colors = img_np.reshape(-1, 3)
valid = np.isfinite(points).all(axis=1)
# if you have mask: valid &= (mask.reshape(-1) > 0)

points = points[valid]
colors = colors[valid]

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points.astype(np.float64))
pcd.colors = o3d.utility.Vector3dVector(colors.astype(np.float64))

o3d.visualization.draw_geometries([pcd])